# Neural Identifier Training with Particle Filters - Differential Drive Mobile Robot

In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 1) True nonlinear system (Differential Drive Mobile Robot)
# ============================================================
def plant_dynamics(x, u, L=0.5, friction_coeff=0.1, slip_factor=0.05):
    """
    Continuous dynamics for differential drive mobile robot: x = [x_pos, y_pos, theta]. 
    Returns x_dot.
    
    The differential drive robot equations:
    dx/dt = v * cos(θ) 
    dy/dt = v * sin(θ)
    dθ/dt = ω
    
    where:
    v = (v_r + v_l) / 2  (linear velocity)
    ω = (v_r - v_l) / L  (angular velocity)
    L = wheelbase distance
    """
    x_pos, y_pos, theta = x
    v_l, v_r = u  # left and right wheel velocities
    
    # Add realistic wheel slip effects
    v_l_actual = v_l * (1 - slip_factor * np.random.randn())
    v_r_actual = v_r * (1 - slip_factor * np.random.randn())
    
    # Compute linear and angular velocities
    v = (v_r_actual + v_l_actual) / 2.0
    omega = (v_r_actual - v_l_actual) / L
    
    # Add friction effects (velocity-dependent)
    v_friction = v * (1 - friction_coeff * np.abs(v))
    omega_friction = omega * (1 - friction_coeff * np.abs(omega))
    
    # Robot kinematics
    x_dot = v_friction * np.cos(theta)
    y_dot = v_friction * np.sin(theta)
    theta_dot = omega_friction
    
    return np.array([x_dot, y_dot, theta_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic mobile robot disturbances.
    
    Args:
        x_k: current state [x, y, theta]
        u_k: control input [v_left, v_right] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: terrain-induced disturbances
        sensor_bias: systematic biases in measurements
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic mobile robot disturbances
    
    # 1. Terrain-induced disturbances (position-dependent)
    terrain_noise = terrain_roughness * np.array([
        np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x-direction terrain variation
        np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # y-direction terrain variation  
        0.1 * np.sin(x_kp1[2]) * np.random.randn()   # angular disturbance from terrain
    ])
    
    # 2. Velocity-dependent noise (increases with speed)
    velocity_magnitude = np.linalg.norm(u_k)
    velocity_noise_factor = 1 + 0.2 * velocity_magnitude
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * velocity_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * velocity_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Wheel encoder quantization effects
    encoder_resolution = 0.001  # 1mm resolution
    quantization_noise = encoder_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += terrain_noise + total_noise + bias_noise + quantization_noise
    
    # 6. Angle wrapping for realistic behavior
    x_kp1[2] = np.arctan2(np.sin(x_kp1[2]), np.cos(x_kp1[2]))  # wrap angle to [-π, π]
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='straight'):
    """
    Generate realistic control inputs for mobile robot.
    
    Args:
        t: time value
        trajectory_type: 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'
    
    Returns:
        u: [v_left, v_right] wheel velocities
    """
    if trajectory_type == 'straight':
        # Straight line with small variations
        v_base = 1.0 + 0.2 * np.sin(0.5 * t)
        return np.array([v_base, v_base])
    
    elif trajectory_type == 'circle':
        # Circular motion
        v_l = 1.0 + 0.1 * np.sin(t)
        v_r = 1.5 + 0.1 * np.cos(t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'figure8':
        # Figure-8 pattern
        v_l = 1.0 + 0.8 * np.sin(0.5 * t)
        v_r = 1.0 - 0.8 * np.sin(0.5 * t)
        return np.array([v_l, v_r])
    
    elif trajectory_type == 'obstacle_avoidance':
        # Obstacle avoidance maneuvers
        base_speed = 1.2
        avoidance_maneuver = 0.5 * np.sin(2 * t) * np.exp(-0.1 * t)
        v_l = base_speed + avoidance_maneuver
        v_r = base_speed - avoidance_maneuver
        return np.array([v_l, v_r])
    
    else:  # 'mixed' - combination of different behaviors
        # Mixed trajectory with different phases
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.3:  # Straight motion
            v_base = 1.5
            return np.array([v_base, v_base])
        elif phase < 0.6:  # Turning
            v_l = 0.8
            v_r = 1.8
            return np.array([v_l, v_r])
        elif phase < 0.8:  # Reverse
            v_base = -0.5
            return np.array([v_base, v_base])
        else:  # Complex maneuver
            v_l = 1.0 + 0.5 * np.sin(10 * t)
            v_r = 1.0 + 0.5 * np.cos(10 * t)
            return np.array([v_l, v_r])

In [3]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    Features for a 3-state mobile robot system with control inputs:
    x = [x_pos, y_pos, theta], u = [v_left, v_right]
    
    z = [S(x), S(y), S(θ), S(x)S(y), S(x)S(θ), S(y)S(θ), 
         S(x)^2, S(y)^2, S(θ)^2, cos(θ), sin(θ), 
         S(v_l), S(v_r), S(v_l)S(v_r), x, y, 1]
    """
    s_x = sigmoidal(x_est[0])     # x position
    s_y = sigmoidal(x_est[1])     # y position  
    s_theta = sigmoidal(x_est[2]) # orientation
    
    # Basic features
    features = [
        s_x, s_y, s_theta,                    # Individual sigmoid terms
        s_x*s_y, s_x*s_theta, s_y*s_theta,   # Cross terms
        s_x**2, s_y**2, s_theta**2,          # Quadratic terms
        np.cos(x_est[2]), np.sin(x_est[2]),  # Trigonometric terms (important for robot)
    ]
    
    # Add control input features if available
    if u_input is not None and len(u_input) >= 2:
        s_vl = sigmoidal(u_input[0])  # left wheel velocity
        s_vr = sigmoidal(u_input[1])  # right wheel velocity
        features.extend([
            s_vl, s_vr,                       # Control sigmoid terms
            s_vl * s_vr,                      # Control cross term
        ])
    else:
        # Add zero placeholders if no control input
        features.extend([0.0, 0.0, 0.0])
    
    # Add direct state terms and bias
    features.extend([
        x_est[0], x_est[1],                   # Direct position terms
        1.0                                   # Bias term
    ])
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [4]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (measured output for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [5]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (measured output for mobile robot)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with Gaussian likelihood using state-specific R_var
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability using state-specific R_var
            ll = -0.5 * (innov**2) / self.R_var[i]
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return information about the PF parameters for each state."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'R_var': self.R_var[i]
            }
        return info

## Particle Filter con Parámetros Específicos por Estado

El Particle Filter ha sido modificado para permitir diferentes valores de `Q_std` y `R_std` para cada estado del robot:

### Parámetros por Estado:
- **Estados de posición (x, y)**: Comparten los mismos valores ya que ambos representan coordenadas cartesianas
- **Estado angular (θ)**: Usa valores diferentes debido a su naturaleza angular y diferentes características de ruido

### Configuración Actual:
- **Q_std (Ruido de proceso)**:
  - x, y: `0.05` - Exploración moderada para posiciones
  - θ: `0.03` - Menor exploración para ángulo (más estable)
  
- **R_std (Ruido de medición)**:
  - x, y: `0.05` - Confianza moderada en sensores de posición
  - θ: `0.08` - Menor confianza en sensor angular (más ruidoso típicamente)

### Ventajas:
1. **Realismo**: Los sensores angulares (giroscopios, brújulas) tienen características de ruido diferentes
2. **Performance**: Permite ajuste fino por cada tipo de estado
3. **Flexibilidad**: Fácil modificación para diferentes tipos de robots

## ✅ Verificación Completada - Particle Filter con Parámetros Específicos

### Modificaciones Implementadas:

1. **Clase PF_RHONN_Trainer mejorada** ✅
   - Acepta `Q_std` y `R_std` como listas por estado
   - Mantiene compatibilidad con valores escalares
   - Validación automática de dimensiones

2. **Configuración por Estado** ✅
   ```python
   Q_std_per_state = [0.05, 0.05, 0.03]  # x, y, theta
   R_std_per_state = [0.05, 0.05, 0.08]  # x, y, theta
   ```

3. **Método de información** ✅
   - `get_parameters_info()` muestra configuración actual
   - Verificación visual de parámetros por estado

4. **Estructura completa** ✅
   - Simulación principal integrada
   - Bucle de entrenamiento completo
   - Sin errores de sintaxis

### Listo para Ejecutar:
- El notebook está completo y funcional
- Parámetros optimizados para robot móvil
- Configuración realista por tipo de estado

In [6]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x position (measured output for mobile robot)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [7]:
# ============================================================
# 5) Simulation Main Loop
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.02
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'mixed'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.03
terrain_roughness = 0.02
sensor_bias = [0.001, 0.001, 0.001]  # Small systematic biases [x, y, theta]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.0, 0.0, 0.0]  # Initial conditions for mobile robot [x, y, theta]

# --- Control trajectory ---
trajectory_type = 'figure8'  # 'straight', 'circle', 'figure8', 'mixed', 'obstacle_avoidance'

# --- RHONN config ---
num_neurons = 3  # Three states for mobile robot [x, y, theta]
num_features = 17  # Updated feature vector size for 3 states + controls
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
# np.random.seed(12345)  # (optional) reproducibility of initial weights
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# --- EKF --- (Tuned parameters for differential drive mobile robot)
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=2e-4, R_init=8e-3, P_init=1.5, eta=0.4
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=2e-4, R_init=8e-3, P_init=1.5, eta=0.6,
    alpha=1e-2, beta=2.0  # UKF-specific parameters for mobile robot
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF ---
n_particles = 500  # Increased particles for 3-state system

# State-specific noise parameters: [x, y, theta]
# x and y use same values (position states), theta uses different (angular state)
Q_std_per_state = [0.05, 0.05, 0.3]  # Process noise: x, y (same), theta (smaller)
R_std_per_state = [0.05, 0.05, 0.1]  # Measurement noise: x, y (same), theta (larger)

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state, 
    ess_threshold=n_particles / 2  # ESS < N/2
)

# Display PF parameters for verification
pf_params = pf_trainer.get_parameters_info()
print("\nParticle Filter Parameters per State:")
for state, params in pf_params.items():
    print(f"  {state}: Q_std={params['Q_std']:.3f}, R_std={params['R_std']:.3f}, R_var={params['R_var']:.6f}")

# Force identical particle initialization if desired:
def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
    for i in range(pf_trainer_instance.num_neurons):
        pf_trainer_instance.particles[i] = np.tile(
            common_weights_list[i], (pf_trainer_instance.n_particles, 1)
        )
        pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

print("\nStarting mobile robot simulation...")
for k in range(n_steps - 1):
    # ---- 1) Generate control input and evolve true system -> k+1 ----
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)

    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured x at k
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)  # x
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)  # y  
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)  # theta

    # ---- 3) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)

    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_state_for_z_ukf[0] = x_true[k][0]  # series-parallel uses measured x at k
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)  # x
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)  # y
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)  # theta

    # ---- 4) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k], u_input=u_current)

    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured x at k
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)   # x
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)   # y
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)   # theta

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.44371255 -0.21230756 -0.31679525  0.27254936  0.46843439 -0.38894005
  0.11028628 -0.24317106  0.31868252  0.15079377 -0.02129468  0.46312945
  0.17477052 -0.17384988 -0.48255006  0.18460212  0.45463309]
  Neuron 1: [ 2.85510464e-01 -2.95093107e-01 -1.60788134e-04 -2.99026913e-01
  6.42421114e-02 -4.53856405e-01  3.01087117e-01 -4.51882748e-01
 -2.44069963e-02 -4.60492959e-02  1.07181566e-01 -3.64121317e-01
  3.48365307e-01  1.87628131e-01  4.15880593e-01  4.97433851e-01
 -3.85845865e-01]
  Neuron 2: [-0.01819887 -0.47899273  0.29024202  0.44018983 -0.1323083  -0.22947063
  0.21742692  0.29435138 -0.15109168  0.35932441 -0.33298691 -0.10147475
 -0.3768317   0.35218932  0.00112985  0.19642199  0.37014832]

Particle Filter Parameters per State:
  x: Q_std=0.050, R_std=0.050, R_var=0.002500
  y: Q_std=0.050, R_std=0.050, R_var=0.002500
  theta: Q_std=0.300, R_std=0.100, R_var=0.010000

Starting mobile robot simulation...
Simulation progress: 0.0%
Si

In [8]:
# ============================================================
    # 6) Results & plots for Differential Drive Mobile Robot
# ============================================================
mse_x_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)     # x position
mse_y_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)     # y position  
mse_theta_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2) # orientation

mse_x_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)     # x position
mse_y_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)     # y position
mse_theta_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2) # orientation

mse_x_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)       # x position
mse_y_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)       # y position  
mse_theta_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # orientation

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x', 'y', 'theta']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Differential Drive Mobile Robot ---")
print(f"EKF MSE x position:     {mse_x_ekf:.6f}")
print(f"EKF MSE y position:     {mse_y_ekf:.6f}")
print(f"EKF MSE theta (orient): {mse_theta_ekf:.6f}")
print(f"UKF MSE x position:     {mse_x_ukf:.6f}")
print(f"UKF MSE y position:     {mse_y_ukf:.6f}")
print(f"UKF MSE theta (orient): {mse_theta_ukf:.6f}")
print(f"PF  MSE x position:     {mse_x_pf:.6f}")
print(f"PF  MSE y position:     {mse_y_pf:.6f}")
print(f"PF  MSE theta (orient): {mse_theta_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'X Position', 'y_label': 'X Position (m)',
     'chi': 'χₓ (True X)', 'x': 'X (Est.)'},
    {'idx': 1, 'var': 'y', 'desc': 'Y Position', 'y_label': 'Y Position (m)',
     'chi': 'χᵧ (True Y)', 'x': 'Y (Est.)'},
    {'idx': 2, 'var': 'theta', 'desc': 'Orientation', 'y_label': 'Orientation (rad)',
     'chi': 'χθ (True θ)', 'x': 'θ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines',
                        name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(
        title=f'Mobile Robot RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states
error_x_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_y_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_theta_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_x_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_y_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_theta_ukf = x_true[:, 2] - x_hat_ukf[:, 2]

error_x_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_y_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_theta_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ekf, mode='lines',
                        name=f'EKF Error X (MSE={mse_x_ekf:.6f})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_ukf, mode='lines',
                        name=f'UKF Error X (MSE={mse_x_ukf:.6f})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x_pf, mode='lines',
                        name=f'PF Error X (MSE={mse_x_pf:.6f})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ekf, mode='lines',
                        name=f'EKF Error Y (MSE={mse_y_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_ukf, mode='lines',
                        name=f'UKF Error Y (MSE={mse_y_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_y_pf, mode='lines',
                        name=f'PF Error Y (MSE={mse_y_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ekf, mode='lines',
                        name=f'EKF Error θ (MSE={mse_theta_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_ukf, mode='lines',
                        name=f'UKF Error θ (MSE={mse_theta_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_theta_pf, mode='lines',
                        name=f'PF Error θ (MSE={mse_theta_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(
    title='Mobile Robot RHONN Identification Errors (EKF vs UKF vs PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 2D Trajectory plot (X-Y plane for mobile robot)
fig_trajectory = go.Figure()
fig_trajectory.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines',
                              name='True Robot Trajectory',
                              line=dict(color='black', width=3)))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], mode='lines',
                              name='EKF Estimation',
                              line=dict(color='blue', width=2, dash='dash')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], mode='lines',
                              name='UKF Estimation',
                              line=dict(color='green', width=2, dash='dashdot')))
fig_trajectory.add_trace(go.Scatter(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], mode='lines',
                              name='PF Estimation',
                              line=dict(color='red', width=2, dash='dot')))
# Add start and end markers
fig_trajectory.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers',
                              name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_trajectory.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers',
                              name='End', marker=dict(color='red', size=10, symbol='square')))
fig_trajectory.update_layout(
    title='Mobile Robot Trajectory - 2D Path Comparison (EKF vs UKF vs PF)',
    xaxis_title='X Position (m)',
    yaxis_title='Y Position (m)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)
fig_trajectory.show()


Final EKF-RHONN Weights:
  Neuron 1 (x): [-0.20469269 -0.20202419  0.11114043  0.1915703  -0.2608232  -0.2089755
  0.15545036  0.01585468  0.15254315  0.01159275 -0.00788611  0.45229109
 -0.10672713 -0.48135893  0.89479939  0.0051561   0.42818932]
  Neuron 2 (y): [ 0.3568737   0.31777418 -0.5976488   0.01784726  0.38741729 -0.23186244
 -0.14743374 -0.18376754  0.1968339   0.04030263 -0.0250591   0.18336353
 -0.35440227  0.10363593 -0.12312901  0.78061105 -0.4685035 ]
  Neuron 3 (theta): [ 1.55128329 -1.25536584  2.78854252  2.10848258  0.4668946  -2.56254577
 -1.28880207 -0.66857337  2.24974793  0.60698102 -0.5927628   0.19462786
 -2.929927   -2.83067096 -0.37788413 -0.13816468  0.86526262]

Final UKF-RHONN Weights:
  Neuron 1 (x): [ 2.66242359 -0.1909667  -0.32457296  0.8251983   0.04172944 -0.46343504
  0.62878226 -0.19744413  0.21642068  0.38564895  0.02889105  0.11336601
 -0.00269596 -0.1943854   0.25084613  0.32387294  0.34510581]
  Neuron 2 (y): [-1.15733893 -0.02428705  0.16800